# Tic em Trilhas - Construção de Agentes Inteligentes com IA
## M1A4 — Agentes de Pesquisa com LangGraph + Tavily
Nesta aula montamos um agente ReAct orientado a buscas, combinando LangGraph, ChatGPT e a ferramenta Tavily para responder perguntas que exigem informações atualizadas.

### Inicialização do ambiente e bibliotecas
Carregamos variáveis de ambiente com `dotenv` e importamos LangChain/LangGraph. Esses módulos dão suporte ao encadeamento de estados e à criação do agente que combinará LLM com ferramentas externas (como o mecanismo de busca Tavily).

In [2]:
from dotenv import load_dotenv
import os
_ = load_dotenv()
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults

c:\Users\MahyaraParaquett\Documents\PUC IA\IA_Python_training\Trilha3-Modulo1\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


### Registrando a ferramenta de busca
Aqui conectamos o Tavily ao agente. Ele funciona como “olhos na web”, permitindo que o modelo consulte fontes atualizadas quando precisar responder perguntas que exigem fatos recentes.

In [3]:
tool = TavilySearchResults(max_results=3, tavily_api_key=os.getenv("TAVILY_SEARCH_API"))

C:\Users\MahyaraParaquett\AppData\Local\Temp\ipykernel_22880\47839059.py:1: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tool = TavilySearchResults(max_results=3, tavily_api_key=os.getenv("TAVILY_SEARCH_API"))


### Estado compartilhado entre nós do grafo
O `AgentState` encapsula o histórico de mensagens. O uso de `Annotated[..., operator.add]` indica ao LangGraph que novas mensagens são concatenadas automaticamente a cada iteração.

In [4]:
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]


### Montando o agente com LangGraph
Construímos um grafo com dois nós principais:
1. `llm`: consulta o modelo para decidir o próximo passo.
2. `action`: executa as ferramentas solicitadas via `tool_calls`.
As arestas condicionais garantem o ciclo “modelo → ferramenta → modelo” até que não haja mais ações pendentes.

In [5]:
class Agent:

    def __init__(self, model, tools, system=""):
        self.system = system
        graph = StateGraph(AgentState)
        graph.add_node("llm", self.call_openai)
        graph.add_node("action", self.take_action)
        graph.add_conditional_edges(
            "llm",
            self.exists_action,
            {True: "action", False: END}
        )
        graph.add_edge("action", "llm")
        graph.set_entry_point("llm")
        self.graph = graph.compile()
        self.tools = {t.name: t for t in tools}
        self.model = model.bind_tools(tools)

    def exists_action(self, state: AgentState):
        result = state['messages'][-1]
        return len(result.tool_calls) > 0

    def call_openai(self, state: AgentState):
        messages = state['messages']
        if self.system:
            messages = [SystemMessage(content=self.system)] + messages
        message = self.model.invoke(messages)
        return {'messages': [message]}

    def take_action(self, state: AgentState):
        tool_calls = state['messages'][-1].tool_calls
        results = []
        for t in tool_calls:
            print(f"Chamando: {t}")
            if not t['name'] in self.tools:      # verificar nome de ferramenta incorreto do LLM
                print("\n ....nome de ferramenta incorreto....")
                result = "nome de ferramenta incorreto, tente novamente"  # instruir LLM a tentar novamente
            else:
                result = self.tools[t['name']].invoke(t['args'])
            results.append(ToolMessage(tool_call_id=t['id'], name=t['name'], content=str(result)))
        print("De volta ao modelo!")
        return {'messages': results}

### Prompt de sistema focado em pesquisa
O texto abaixo orienta o LLM a agir como pesquisador disciplinado: só buscar quando fizer sentido, explicar múltiplas buscas e usar o motor como apoio antes de responder.

In [6]:
prompt = """Você é um assistente de pesquisa inteligente. Use o motor de busca para procurar informações. \
Você pode fazer múltiplas chamadas (juntas ou em sequência). \
Só procure informações quando tiver certeza do que quer. \
Se precisar procurar algumas informações antes de fazer uma pergunta de acompanhamento, você pode fazer isso!
"""

### Instanciando o modelo com ferramentas ligadas
Enlaçamos o `ChatOpenAI` ao grafo e passamos a lista de ferramentas disponíveis. A partir daqui, cada mensagem do humano percorre automaticamente o fluxo definido.

In [7]:
model = ChatOpenAI(
    api_key= os.environ.get("OPENROUTER_API_KEY"),
    base_url='https://openrouter.ai/api/v1',
    model="openai/gpt-4o-mini",
     temperature=0,
    streaming=True
    )
abot = Agent(model, [tool], system=prompt)

### Exemplo 1 — Pergunta única com uma chamada de ferramenta
Testamos o agente com uma pergunta meteorológica. Observe no output como o LLM pede o Tavily, aguarda o `ToolMessage` e só então compõe a resposta final para o usuário.

In [8]:
messages = [HumanMessage(content="Qual é o clima no Rio de Janeiro?")]
result = abot.graph.invoke({"messages": messages})
print("Resultado completo:")
print(result)
print("\nResposta final:")
print(result['messages'][-1].content)

Chamando: {'name': 'tavily_search_results_json', 'args': {'query': 'clima no Rio de Janeiro'}, 'id': 'call_6SWRdKuc6rqDlkMgTodBlYxi', 'type': 'tool_call'}
De volta ao modelo!
Resultado completo:
{'messages': [HumanMessage(content='Qual é o clima no Rio de Janeiro?', additional_kwargs={}, response_metadata={}), AIMessage(content='', additional_kwargs={}, response_metadata={'finish_reason': 'tool_callstool_calls', 'model_name': 'openai/gpt-4o-miniopenai/gpt-4o-mini', 'system_fingerprint': 'fp_83e2dd34fcfp_83e2dd34fc', 'model_provider': 'openai'}, id='lc_run--019db13d-08d2-78a1-bc59-45d9b3964100', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'clima no Rio de Janeiro'}, 'id': 'call_6SWRdKuc6rqDlkMgTodBlYxi', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 148, 'output_tokens': 23, 'total_tokens': 171, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}), ToolMessage(content="[{'tit

### Exemplo 2 — Perguntas encadeadas e múltiplas buscas
Agora o agente precisa combinar dois fatos (campeão da Copa de 2022 + PIB do país). O grafo permite múltiplas `tool_calls` em sequência, mostrando como o ReAct se generaliza para fluxos de raciocínio mais longos.

In [9]:
query = "Quem ganhou a Copa do Mundo de 2022? Qual é o PIB desse país? Responda cada pergunta." 
messages = [HumanMessage(content=query)]
abot = Agent(model, [tool], system=prompt)
result = abot.graph.invoke({"messages": messages})
print(result['messages'][-1].content)

Chamando: {'name': 'tavily_search_results_json', 'args': {'query': 'Copa do Mundo 2022 vencedor'}, 'id': 'call_CsNeOyLeWpLpluK9L4EPAenR', 'type': 'tool_call'}
Chamando: {'name': 'tavily_search_results_json', 'args': {'query': 'PIB Argentina 2023'}, 'id': 'call_bmKcmJYeEO2CYzGvDbKddpri', 'type': 'tool_call'}
De volta ao modelo!
1. **Quem ganhou a Copa do Mundo de 2022?**
   A Argentina foi a vencedora da Copa do Mundo de 2022, conquistando o título pela terceira vez em sua história. A final foi disputada contra a França e terminou empatada em 3 a 3, com a Argentina vencendo por 4 a 2 na disputa de pênaltis. [Mais informações aqui](https://pt.wikipedia.org/wiki/Final_da_Copa_do_Mundo_FIFA_de_2022).

2. **Qual é o PIB da Argentina?**
   O PIB da Argentina foi de aproximadamente 646,08 bilhões de dólares em 2023. [Mais detalhes aqui](https://es.tradingeconomics.com/argentina/gdp).


In [10]:
query = "Quala a capital do Canadá?" 
messages = [HumanMessage(content=query)]
abot = Agent(model, [tool], system=prompt)
result = abot.graph.invoke({"messages": messages})
print(result['messages'][-1].content)

A capital do Canadá é Ottawa.
